# Импорты

In [1]:
from enum import Enum
from tokenizers import Tokenizer
from torch.utils.data.dataset import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tokenizers.models import BPE, Unigram
from torch.utils.data import DataLoader
import torch

/home/misha/Desktop/knowledge_distillation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Инициализация токенизаторов

In [2]:
model_name_1 = "LiquidAI/LFM2.5-230M"
tokenizer_1 = Tokenizer.from_pretrained(model_name_1)
print(f'Длина словаря {model_name_1}: {tokenizer_1.get_vocab_size()}')
print(f'Модель токенизатора {model_name_1}: {tokenizer_1.model}')

Длина словаря LiquidAI/LFM2.5-230M: 64402
Модель токенизатора LiquidAI/LFM2.5-230M: BPE(dropout=None, unk_token=None, continuing_subword_prefix=None, end_of_word_suffix=None, fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={"<|pad|>":0, "<|startoftext|>":1, "<|endoftext|>":2, "<|fim_pre|>":3, "<|fim_mid|>":4, ...}, merges=[("Ċ", "Ċ"), ("Ċ", "ĊĊ"), ("ĊĊ", "Ċ"), ("Ċ", "ĊĊĊ"), ("ĊĊ", "ĊĊ"), ...])


In [3]:
model_name_2 = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer_2 = Tokenizer.from_pretrained(model_name_2)
print(f'Длина словаря {model_name_2}: {tokenizer_2.get_vocab_size()}')
print(f'Модель токенизатора {model_name_2}: {tokenizer_2.model}')

Длина словаря Qwen/Qwen2.5-0.5B-Instruct: 151665
Модель токенизатора Qwen/Qwen2.5-0.5B-Instruct: BPE(dropout=None, unk_token=None, continuing_subword_prefix="", end_of_word_suffix="", fuse_unk=False, byte_fallback=False, ignore_merges=False, vocab={"!":0, """:1, "#":2, "$":3, "%":4, ...}, merges=[("Ġ", "Ġ"), ("ĠĠ", "ĠĠ"), ("i", "n"), ("Ġ", "t"), ("ĠĠĠĠ", "ĠĠĠĠ"), ...])


# Инициализация моделей

In [4]:
tokenizer_model_1 = AutoTokenizer.from_pretrained(model_name_1)
model_1 = AutoModelForCausalLM.from_pretrained(model_name_1,
                                               torch_dtype=torch.float16,
                                               device_map="auto")
model_1.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 132/132 [00:00<00:00, 1407.96it/s]


Lfm2ForCausalLM(
  (model): Lfm2Model(
    (embed_tokens): Embedding(65536, 1024, padding_idx=0)
    (layers): ModuleList(
      (0-1): 2 x Lfm2DecoderLayer(
        (conv): Lfm2ShortConv(
          (conv): Conv1d(1024, 1024, kernel_size=(3,), stride=(1,), padding=(2,), groups=1024, bias=False)
          (in_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=False)
        )
        (feed_forward): Lfm2MLP(
          (w1): Linear(in_features=1024, out_features=2560, bias=False)
          (w3): Linear(in_features=1024, out_features=2560, bias=False)
          (w2): Linear(in_features=2560, out_features=1024, bias=False)
        )
        (operator_norm): Lfm2RMSNorm((1024,), eps=1e-05)
        (ffn_norm): Lfm2RMSNorm((1024,), eps=1e-05)
      )
      (2): Lfm2DecoderLayer(
        (self_attn): Lfm2Attention(
          (q_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (k_proj): Li

In [5]:
tokenizer_model_2 = AutoTokenizer.from_pretrained(model_name_2)
model_2 = AutoModelForCausalLM.from_pretrained(model_name_2,
                                               torch_dtype=torch.float16,
                                               device_map="auto")
model_2.eval()

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1397.92it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

# Предобработка данных

In [6]:
class TokenizerModel(Enum):
    BPE = "BPE"
    WordPiece = "WordPiece"
    Unigram = "Unigram"

In [7]:
exact_match_dict = {}
model_1_exact_tokens = []
model_2_exact_tokens = []

non_exact_tokens = []
vocab_1 = tokenizer_1.get_vocab()
vocab_2 = tokenizer_2.get_vocab()


if isinstance(tokenizer_1.model, BPE):
    if isinstance(tokenizer_2.model, Unigram):
        for key in vocab_1.keys():
            key_candidate = key
            if key[0] == 'Ġ':
                key = key.replace('Ġ', '▁')
            if key in vocab_2.keys():
                exact_match_dict[vocab_1[key_candidate]] = vocab_2[key]
                model_1_exact_tokens.append(vocab_1[key_candidate])
                model_2_exact_tokens.append(vocab_2[key])
    if isinstance (tokenizer_2.model, BPE):
        for key in vocab_1.keys():
            if key in vocab_2.keys():
                exact_match_dict[vocab_1[key]] = vocab_2[key]
                model_1_exact_tokens.append(vocab_1[key])
                model_2_exact_tokens.append(vocab_2[key])


exact_match_list = [(token_1, token_2) for token_1, token_2 in exact_match_dict.items()]

In [ ]:
for exact_match_list

[(34148, 80015),
 (19187, 77131),
 (35237, 12115),
 (17657, 24654),
 (59122, 66393),
 (58987, 50906),
 (27382, 99636),
 (60308, 51261),
 (57845, 51760),
 (5937, 6430),
 (61337, 71376),
 (19165, 15542),
 (40760, 45950),
 (22702, 31441),
 (52637, 19202),
 (32614, 87796),
 (18159, 23522),
 (4680, 7135),
 (52539, 36972),
 (31937, 67731),
 (3518, 26232),
 (20777, 131692),
 (11944, 19179),
 (1382, 968),
 (4430, 15326),
 (25315, 28193),
 (1500, 24725),
 (51208, 43862),
 (6615, 18808),
 (13663, 6119),
 (38913, 9903),
 (42429, 28668),
 (14513, 57967),
 (23948, 4506),
 (13269, 8189),
 (4089, 36407),
 (11034, 45131),
 (63748, 38390),
 (10661, 5372),
 (21618, 13510),
 (46270, 38426),
 (36741, 28436),
 (15198, 78361),
 (36467, 42001),
 (30690, 21018),
 (1690, 2565),
 (26272, 132169),
 (5310, 75375),
 (14515, 19609),
 (7465, 66024),
 (46246, 104316),
 (21640, 16090),
 (47901, 10944),
 (61977, 48127),
 (33088, 94677),
 (32119, 6890),
 (22484, 141385),
 (7016, 1397),
 (2336, 28120),
 (25151, 24416),
 

In [9]:
for value in vocab_1.values():
    if value == 1:
        print('sadas')

sadas


In [10]:
exact_match_dict

{34148: 80015,
 19187: 77131,
 35237: 12115,
 17657: 24654,
 59122: 66393,
 58987: 50906,
 27382: 99636,
 60308: 51261,
 57845: 51760,
 5937: 6430,
 61337: 71376,
 19165: 15542,
 40760: 45950,
 22702: 31441,
 52637: 19202,
 32614: 87796,
 18159: 23522,
 4680: 7135,
 52539: 36972,
 31937: 67731,
 3518: 26232,
 20777: 131692,
 11944: 19179,
 1382: 968,
 4430: 15326,
 25315: 28193,
 1500: 24725,
 51208: 43862,
 6615: 18808,
 13663: 6119,
 38913: 9903,
 42429: 28668,
 14513: 57967,
 23948: 4506,
 13269: 8189,
 4089: 36407,
 11034: 45131,
 63748: 38390,
 10661: 5372,
 21618: 13510,
 46270: 38426,
 36741: 28436,
 15198: 78361,
 36467: 42001,
 30690: 21018,
 1690: 2565,
 26272: 132169,
 5310: 75375,
 14515: 19609,
 7465: 66024,
 46246: 104316,
 21640: 16090,
 47901: 10944,
 61977: 48127,
 33088: 94677,
 32119: 6890,
 22484: 141385,
 7016: 1397,
 2336: 28120,
 25151: 24416,
 53299: 41112,
 22735: 9562,
 12358: 27069,
 21923: 26753,
 25198: 43611,
 22901: 30125,
 22554: 90130,
 15008: 30691,
 6

In [11]:
non_exact_model_1 = {}
non_exact_model_2 = {}
embedding_layer_1 = model_1.get_input_embeddings()
embedding_layer_2 = model_2.get_input_embeddings()

for token, idx in vocab_1.items():
    if idx not in model_1_exact_tokens:
        non_exact_model_1[idx] = embedding_layer_1.weight[idx]

for token, idx in vocab_2.items():
    if idx not in model_2_exact_tokens:
        non_exact_model_2[idx] = embedding_layer_2.weight[idx]

In [12]:
embedding_layer_1 = model_1.get_input_embeddings()
embedding_layer_2 = model_2.get_input_embeddings()
dataset = []

for token_1, token_2 in exact_match_list:
    embed_1 = embedding_layer_1.weight[token_1]
    print(embed_1.shape)
    embed_2 = embedding_layer_2.weight[token_2]
    # print(embed_2.shape)
    dataset.append((embed_1, embed_2))

torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([1024])
torch.Size([

In [13]:
dataset_non_exact_1 = {}
dataset_non_exact_2 = {}

for token in non_exact_tokens:
    embed_1 = embedding_layer_1.weight[token]
    dataset_non_exact_1[token] = embed_1

# Построение пайплайна загрузки данных в модель

In [14]:
class Kind(Enum):
    train = "train"
    test = "test"

TRAIN_RATIO = 0.8

In [15]:
class ExactMatchDataset(Dataset):
    def __init__(self, dataset: list[tuple], train_ratio: float, kind: Kind):
        self.dataset = self._train_test_split(dataset=dataset, train_ratio=train_ratio, kind=kind)
    
    def _train_test_split(self, dataset: list[tuple], train_ratio: float, kind: str):
        upper_limit = int(len(dataset)*train_ratio)
        if kind == Kind.train:
            return dataset[:upper_limit]
        elif kind == Kind.test:
            return dataset[upper_limit:]
        else:
            raise ValueError("Выбран неправильный тип датасета: возможен только train или test")

    def __getitem__(self, idx):
        model_1_embed = self.dataset[idx][0] # input embed
        model_2_embed = self.dataset[idx][1] # output embed
        return model_1_embed, model_2_embed
    
    def __len__(self):
        return len(self.dataset)

In [16]:
train = ExactMatchDataset(dataset, TRAIN_RATIO, Kind.train)
valid = ExactMatchDataset(dataset, TRAIN_RATIO, Kind.test)

train_dataloader = DataLoader(train, batch_size=64, shuffle=True)
valid_dataloader = DataLoader(valid, batch_size=64, shuffle=True)

In [17]:
import torch
import torch.nn as nn

class EmbedProjector(nn.Module):
    def __init__(self, input_dim, num_hidden, output_dim):
        super().__init__()
        self.layer_1 = nn.Linear(in_features=input_dim, out_features=num_hidden)
        self.norm_1 = nn.LayerNorm(num_hidden)
        self.activation_1 = nn.ReLU()
        self.layer_2 = nn.Linear(in_features=num_hidden, out_features=output_dim)
        self.skip = nn.Linear(input_dim, output_dim) if input_dim != output_dim else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = self.skip(x)
        h = self.layer_1(x)
        h = self.norm_1(h)
        h = self.activation_1(h)
        h = self.layer_2(h)
        return h + residual

In [18]:
len(train[2][1])

896

In [19]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")
model = EmbedProjector(input_dim=len(train[2][0]), num_hidden=512, output_dim=len(train[2][1])).to(device)

Using cuda device


In [20]:
from tqdm import tqdm
import torch.optim as optim

class ModelManager:
    def __init__(self, 
                 epoch_num: int, 
                 model: EmbedProjector, 
                 train_dataloader: DataLoader, 
                 val_dataloader: DataLoader,
                 device: str):
        self.epoch_num = epoch_num
        self.model = model
        self.model_dtype = self._get_model_dtype(model=model)
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader
        self.device = device
        self.optimizer = optim.Adam(params=model.parameters(), lr=0.001, weight_decay=1e-4)

    def _get_model_dtype(self, model):
        return next(model.parameters()).dtype
    
    def criterion(self, y_predict, y):
        
        cosine_loss = nn.CosineEmbeddingLoss()
        target = torch.ones(
            y_predict.shape[0],
            device=y_predict.device,
            dtype=y_predict.dtype,
        )
        return cosine_loss(y_predict, y, target)

    def train_and_val_loop(self):
        average_train_loss_arr = {}
        average_val_loss_arr = {}
        cosine_metric = nn.CosineSimilarity(dim=-1)
        cosine_arr = []
        for idx, epoch in enumerate(range(self.epoch_num)):
            self.model.train()
            total_loss = 0
            cosine_similarity = 0
            train_tqdm = tqdm(self.train_dataloader)
            for x_train, y_train in train_tqdm:
                x_train = x_train.to(device=self.device,
                                     dtype=self.model_dtype)
                y_train = y_train.to(device=self.device,
                                     dtype=self.model_dtype)
                self.optimizer.zero_grad()
                predict = self.model(x_train)
                loss = self.criterion(y_predict=predict, y=y_train)
                loss.backward()
                self.optimizer.step()
                total_loss += loss.item()*x_train.size(0)

            mean_loss = total_loss/len(self.train_dataloader.dataset)
            average_train_loss_arr[epoch] = mean_loss
            print(f'Эпоха = {epoch+1}\nСредний train_loss = {mean_loss}')

            self.model.eval()
            with torch.inference_mode():
                val_tqdm = tqdm(self.val_dataloader)
                total_loss = 0
                for x_val, y_val in val_tqdm:
                    x_val = x_val.to(device=self.device,
                                    dtype=self.model_dtype)
                    y_val = y_val.to(device=self.device,
                                    dtype=self.model_dtype)
                    predict = self.model(x_val)
                    loss = self.criterion(y_predict=predict, y=y_val)
                    batch_cosine_similarity = cosine_metric(predict, y_val)
                    cosine_similarity += batch_cosine_similarity.sum().item()
                    total_loss += loss.item()*x_val.size(0)
            
            mean_cos = cosine_similarity/len(self.val_dataloader.dataset)
            cosine_arr.append(mean_cos)
            mean_loss = total_loss/len(self.val_dataloader.dataset)
            average_val_loss_arr[epoch] = mean_loss

            print(f'Средний val_loss = {mean_loss}')
            print(f'Среднее косинусное сходство = {mean_cos}')
            if idx != 0:
                if average_val_loss_arr[idx] < average_val_loss_arr[idx-1]:
                    print(f'Качество модели улучшилось на {average_val_loss_arr[idx-1]-average_val_loss_arr[idx]} по лоссу, сохраняем модель')
                    torch.save(self.model.state_dict(), 'embedder_model_qwen_liquid.pth')
                else:
                    print('Модель не оверперформит, заканчиваю обучение...')
                    break
            print('###########################')

            
            
        
        return average_train_loss_arr, average_val_loss_arr

In [30]:
model_manager = ModelManager(epoch_num=25,
                             model=model, 
                             train_dataloader=train_dataloader, 
                             val_dataloader=valid_dataloader,
                             device='cuda')

average_train_loss_arr, average_val_loss_arr = model_manager.train_and_val_loop()

  0%|          | 0/594 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 MiB. GPU 0 has a total capacity of 7.63 GiB of which 54.31 MiB is free. Process 10263 has 3.29 GiB memory in use. Including non-PyTorch memory, this process has 2.58 GiB memory in use. Of the allocated memory 2.37 GiB is allocated by PyTorch, and 10.01 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [21]:
model_filename = 'embedder_model_qwen_liquid.pth'

model_new = EmbedProjector(input_dim=len(train[2][0]), num_hidden=512, output_dim=len(train[2][1])).to(device)
model_new.load_state_dict(
    torch.load(model_filename, map_location="cuda")
)
model_new.eval()

EmbedProjector(
  (layer_1): Linear(in_features=1024, out_features=512, bias=True)
  (norm_1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
  (activation_1): ReLU()
  (layer_2): Linear(in_features=512, out_features=896, bias=True)
  (skip): Linear(in_features=1024, out_features=896, bias=True)
)

In [22]:
vocab_1_reverse = {value: key for key, value in vocab_1.items()}
vocab_2_reverse = {value: key for key, value in vocab_2.items()}

In [23]:
test = list(non_exact_model_1.values())
# print(f'{test[:5]=}')
tensor1 = test[0]
tensor2 = test[1]

print(f'{tensor1=}')
print(f'{tensor1.shape=}')
print(f'{tensor2=}')
print(f'{tensor2.shape=}')

print(f'{torch.stack((tensor1, tensor2), 0)}')
print(f'{torch.stack((tensor1, tensor2), 0).shape}')

tensor1=tensor([-0.0193,  0.0042,  0.0095,  ...,  0.0527, -0.0435, -0.0032],
       device='cuda:0', dtype=torch.float16, grad_fn=<SelectBackward0>)
tensor1.shape=torch.Size([1024])
tensor2=tensor([ 0.0206,  0.0388,  0.0019,  ..., -0.0092,  0.0219,  0.0013],
       device='cuda:0', dtype=torch.float16, grad_fn=<SelectBackward0>)
tensor2.shape=torch.Size([1024])
tensor([[-0.0193,  0.0042,  0.0095,  ...,  0.0527, -0.0435, -0.0032],
        [ 0.0206,  0.0388,  0.0019,  ..., -0.0092,  0.0219,  0.0013]],
       device='cuda:0', dtype=torch.float16, grad_fn=<StackBackward0>)
torch.Size([2, 1024])


In [24]:
# Получаем предсказания от модели по эмбеддингам несовпавших токенов

non_exact_model_1_after_model = {}

for token, embed in non_exact_model_1.items():
    projected_embed = model_new(embed.to(device='cuda', dtype=next(model_new.parameters()).dtype))
    non_exact_model_1_after_model[token] = projected_embed

source_emb = torch.stack(list(non_exact_model_1_after_model.values()))
target_emb = torch.stack(list(non_exact_model_2.values()))

In [25]:
import torch
import torch.nn.functional as F

def running_topk_mean(query, keys, k=10, batch_size=2048):
    """
    Для каждой строки query считает среднее по top-k косинусных сходств
    с keys, не храня полную матрицу сходств.
    query: [N_q, D] (нормализованные)
    keys:  [N_k, D] (нормализованные)
    returns: [N_q] — средняя близость к top-k соседям в keys
    """
    N_q = query.shape[0]
    result = torch.empty(N_q, device=query.device, dtype=query.dtype)

    for start in range(0, N_q, batch_size):
        end = min(start + batch_size, N_q)
        q_batch = query[start:end]                # [B, D]
        sims = q_batch @ keys.T                    # [B, N_k] — временная, но маленькая по B
        topk_vals = sims.topk(k, dim=1).values      # [B, k]
        result[start:end] = topk_vals.mean(dim=1)
        del sims, topk_vals
        torch.cuda.empty_cache()

    return result


def csls_best_match(src_embeds, tgt_embeds, k=10, batch_size=2048):
    src = F.normalize(src_embeds, dim=-1)
    tgt = F.normalize(tgt_embeds, dim=-1)

    # r_S(y): близость каждого tgt-токена к его top-k соседям среди src
    r_tgt = running_topk_mean(tgt, src, k=k, batch_size=batch_size)  # [N_tgt]

    # r_T(x): близость каждого src-токена к его top-k соседям среди tgt
    r_src = running_topk_mean(src, tgt, k=k, batch_size=batch_size)  # [N_src]

    best_idx = torch.empty(src.shape[0], dtype=torch.long, device=src.device)
    best_score = torch.empty(src.shape[0], device=src.device, dtype=src.dtype)

    for start in range(0, src.shape[0], batch_size):
        end = min(start + batch_size, src.shape[0])
        s_batch = src[start:end]                    # [B, D]
        sims = s_batch @ tgt.T                       # [B, N_tgt]
        csls_batch = 2 * sims - r_src[start:end].unsqueeze(1) - r_tgt.unsqueeze(0)
        vals, idx = csls_batch.max(dim=1)
        best_idx[start:end] = idx
        best_score[start:end] = vals
        del sims, csls_batch
        torch.cuda.empty_cache()

    return best_idx, best_score

In [26]:
tokens_1 = list(non_exact_model_1_after_model.keys())
tokens_2 = list(non_exact_model_2.keys())

source_emb = torch.stack([
    non_exact_model_1_after_model[token_id].squeeze(0)
    for token_id in tokens_1
]).detach()

target_emb = torch.stack([
    non_exact_model_2[token_id].squeeze(0)
    for token_id in tokens_2
]).detach()

target_emb = target_emb.to(
    device=source_emb.device,
    dtype=source_emb.dtype,
)

In [29]:
import torch, gc
torch.cuda.empty_cache()
gc.collect()

1241

In [30]:
best_idx, best_score = csls_best_match(
    source_emb,
    target_emb,
    k=10,
    batch_size=256,
)

vocab_mapping = {}

for i, t1 in enumerate(tokens_1):
    target_row = best_idx[i].item()
    t2 = tokens_2[target_row]

    print(
        f"{vocab_1_reverse[t1]} -> "
        f"{vocab_2_reverse[t2]}, "
        f"CSLS={best_score[i].item():.4f}")
    
    vocab_mapping[t1] = t2

Ġdrie -> _three, CSLS=0.0363
04 -> èİ·æī¹, CSLS=-0.1238
ccan -> ì³ī, CSLS=-0.1103
onnes -> ONES, CSLS=-0.1315
ĠFortun -> fortune, CSLS=-0.1817
Ġ$- -> Ġ-$, CSLS=0.1310
Ġataque -> ĠAttacks, CSLS=0.1208
Ã©ly -> ï®°, CSLS=-0.0753
Ġofficiel -> official, CSLS=0.0102
Unter -> ðĴĢ¸, CSLS=-0.1987
overing -> OVER, CSLS=-0.1428
Ġbereit -> ĠÃ¼berh, CSLS=-0.0216
ÐĳÐ¸ -> _bi, CSLS=-0.1398
ĠRelig -> religious, CSLS=0.0458
Ð±Ð¾ÑĢÐ° -> -selection, CSLS=-0.1920
ĠQuer -> ĠQueries, CSLS=0.2181
ĠZel -> ĉz, CSLS=0.0578
ĠGeist -> ĠSpirits, CSLS=-0.1867
Ġhaj -> ðĿķĻ, CSLS=-0.1868
Ġnac -> Naz, CSLS=-0.1616
Ġgare -> Ġlingerie, CSLS=-0.2025
Ã©met -> ë©Ĥ, CSLS=-0.0458
Ġpochod -> Ġ×ł×Ķ×ł, CSLS=-0.1699
Ġvoul -> ĠdÃ©sir, CSLS=0.0061
Ġitiner -> Ġitinerary, CSLS=0.2619
iÃ¡n -> IÃĬ, CSLS=-0.2005
ÑĤÐ¾Ñĩ -> ð¬Ń, CSLS=-0.1613
atique -> íĴľ, CSLS=-0.1540
ensic -> ï®Ī, CSLS=-0.0975
naio -> aincontri, CSLS=-0.0746
ĠfrancÃ©s -> Ġfrench, CSLS=0.0723
ĠÐĴÐ°Ð» -> âĭ¼, CSLS=-0.1438
itance -> itution, CSLS=-0.1820
ÑĥÐ±Ð»Ð¸ÐºÐ¾Ð² ->

In [38]:
from transformers.generation import GenerationMixin
from transformers.tokenization_utils_tokenizers import TokenizersBackend
from transformers.tokenization_utils_sentencepiece import SentencePieceBackend

class GenerationLoop:
    def __init__(self, 
                 model_1: GenerationMixin,
                 model_2: GenerationMixin,
                 tokenizer_1: TokenizersBackend | SentencePieceBackend,
                 tokenizer_2: TokenizersBackend | SentencePieceBackend,
                 mapping_vocab: dict):
        self.model_1 = model_1
        self.model_2 = model_2
        self.tokenizer_1 = tokenizer_1
        self.tokenizer_2 = tokenizer_2
        self.mapping_vocab = mapping_vocab
    
    def generate(self, input_txt: str):
        input_1 = self.tokenizer_1(input_txt,
                                   return_tensors="pt",).to(self.model_1.device)
        input_2 = self.tokenizer_2(input_txt,
                                   return_tensors="pt",).to(self.model_2.device)
        
        with torch.inference_mode():
            output_ids_1 = self.model_1.generate(**input_1,
                                               max_new_tokens=1, 
                                               do_sample=False,
                                               return_dict_in_generate=True,
                                               output_scores=True,)
            
            output_ids_2 = self.model_2.generate(**input_2,
                                               max_new_tokens=1, 
                                               do_sample=False,
                                               return_dict_in_generate=True,
                                               output_scores=True,)

            next_token_scores_1 = output_ids_1.scores[0]
            next_token_scores_2 = output_ids_2.scores[0]

            next_token_probs_1 = torch.softmax(next_token_scores_1, dim=-1)
            next_token_probs_2 = torch.softmax(next_token_scores_2, dim=-1)

            self.print_probs(next_token_probs_1=next_token_probs_1, next_token_probs_2=next_token_probs_2)
    
    def print_probs(self, next_token_probs_1, next_token_probs_2):
        top_probs_1, top_ids_1 = torch.topk(next_token_probs_1[0], k=10)
        top_probs_2, top_ids_2 = torch.topk(next_token_probs_2[0], k=10)

        for rank, (token_id, prob) in enumerate(zip(top_ids_1, top_probs_1), start=1):
            token_id = token_id.item()
            token_text = self.tokenizer_1.decode([token_id], skip_special_tokens=False)
            print(f"{rank}. token='{token_text}' id={token_id} prob={prob.item():.6f}")
            if token_id in self.mapping_vocab:
                target_id = self.mapping_vocab[token_id]
                print(f'  -> в модели 2: {next_token_probs_2[0][target_id].item():.6f}')
            else:
                print('  -> нет в mapping')

        print('######################################################')
        for rank, (token_id, prob) in enumerate(zip(top_ids_2, top_probs_2), start=1):
            token_id = token_id.item()
            token_text = self.tokenizer_2.decode([token_id], skip_special_tokens=False)
            print(f"{rank}. token='{token_text}' id={token_id} prob={prob.item():.6f}")

In [120]:
vocab_2

{'éħįç½®': 85767,
 'Ġissuer': 54835,
 '-built': 51614,
 'Ø§ÙĦØª': 125962,
 '+"\'': 47187,
 'å¤§ç±³': 110319,
 'Ġshelf': 27645,
 'ä¸°å¯Į': 100733,
 'ĠBreitbart': 55876,
 'Ġelectroly': 72296,
 'ãģĭãĤīãģ®': 129906,
 'éĻĮçĶŁ': 102828,
 'inkle': 35144,
 'ĠCarlo': 57770,
 'ĠrefreshToken': 74541,
 'å¸ĸåŃĲ': 110932,
 'ytic': 69404,
 '_needed': 57426,
 'ðĿķģ': 149992,
 'ĠBaÅŁ': 97505,
 'zd': 48655,
 'cretion': 88690,
 'åİĺç±³': 105887,
 'adro': 89676,
 'takes': 77979,
 'Ð¼ÐµÐ¶Ð´ÑĥÐ½Ð°ÑĢÐ¾Ð´Ð½': 137557,
 '(selector': 46418,
 'jure': 52302,
 'BaseUrl': 71587,
 "Ġ'',": 8981,
 'Ġchar': 1161,
 'ĠpieniÄħd': 142638,
 '_TS': 61793,
 'ìħĳ': 149981,
 'Listing': 52564,
 'ĠAnyObject': 41543,
 'ðŁĮ¡': 148100,
 'ĠÄĳÄĥng': 129418,
 'rawing': 1696,
 'pesan': 96285,
 'Ø®ØµÙĪØµ': 131599,
 'åľºåĲĪ': 108430,
 'usp': 29763,
 'ĠÃ§evir': 140383,
 'åģļä¸ĢäºĽ': 112446,
 'å¡': 58557,
 'kc': 31378,
 'ĠdidReceiveMemoryWarning': 38141,
 'ECT': 5935,
 'Ġmathematic': 20976,
 'ÑĢÐ¾Ð²': 23862,
 'ĠØ§ÙĦØŃØ¯ÙĬØ¯': 141043,
 'Ġà¤':

In [ ]:
class Projector:
    def __init__(self, mapping_vocab: dict,
                 model_1: GenerationMixin,
                 model_2: GenerationMixin,
                 tokenizer_1: TokenizersBackend | SentencePieceBackend,
                 tokenizer_2: TokenizersBackend | SentencePieceBackend,
                 mapping_projection: dict,
                 ):
        self.mapping_vocab = mapping_vocab
        self.model_1 = model_1
        self.model_2 = model_2
        self.tokenizer_1 = tokenizer_1
        self.tokenizer_2 = tokenizer_2
        self.mapping_projection = mapping_projection

    
    def map_input(self, input: str):
        message = [{'role': 'user', 'content': input}]

        text = self.tokenizer_1.apply_chat_template(message,
                                                    tokenize=True,
                                                    return_tensors='pt',
                                                    add_generation_prompt=False)
        
        tokens_1 = text['input_ids'][0]
        strings = self.tokenizer_1.decode(tokens_1, skip_special_tokens=True)

        print(f'Разбивка по токенам: {tokens_1=}')
        print(f'Декодировано в текст: {strings=}')

        projected_arr_mlp = []
        projected_arr_proj = []
        for token in tokens_1:
            print(token)
            try:
                print(f'Токен "{token}" и его НС-отображение {self.mapping_vocab[int(token)]}') 
                print(f'Токен "{token}" и его проекционное-отображение {self.mapping_projection[(self.tokenizer_1.decode(token), int(token))]}') 
            except:
                continue
            projected_arr_mlp.append(self.mapping_vocab[int(token)])
            projected_arr_proj.append(self.mapping_projection[(self.tokenizer_1.decode(token), int(token))])
        

        print(f'{projected_arr_mlp=}')
        print(f'{projected_arr_proj=}')
        projected_arr_proj_new = []
        for elem in projected_arr_proj:
            print(f'{elem=}')
            projected_arr_proj
        print(f'Декодированная строка {self.tokenizer_2.decode(projected_arr_mlp)}')
        print(f'Декодированная строка {self.tokenizer_2.decode(projected_arr_proj)}')

    


In [152]:
# Составление финального mapping-словаря

vocab_mapping_tensors = {}


for key, value in vocab_mapping.items(): # Итерация по смоделлированным проекциям
    vocab_mapping_tensors[key] = value

for elem_1, elem_2 in exact_match_dict.items(): # Итерация по exact-match
    vocab_mapping_tensors[elem_1] = elem_2

In [153]:
import json
import re
import ast

with open("out1.json", encoding="utf-8") as f:
    data = json.load(f)

pattern = re.compile(r"^(\(.+?\)) — (\[.+\])$")

result = {}
for line in data["wp_bpe_comparison_str"]:
    m = pattern.match(line)
    key_str, val_str = m.group(1), m.group(2)
    key = ast.literal_eval(key_str)   # tuple (token, id)
    value = ast.literal_eval(val_str) # list[tuple]
    result[key] = value

In [191]:
vocab_mapping_tensors
vocab_mapping_mlp = {}
counter = 0
for key, value in vocab_mapping_tensors.items():
    print(f'{key=}')
    if key in vocab_1_reverse:
        vocab_mapping_mlp[(vocab_1_reverse[key], key)] = [(vocab_2_reverse[value], value)]
    # if vocab_1_reverse[key] != vocab_2_reverse[value]:
    #     counter += 1

key=38595
key=6326
key=59630
key=20373
key=63474
key=37133
key=46291
key=20521
key=30160
key=35961
key=39199
key=48236
key=20037
key=17524
key=44037
key=36875
key=36424
key=44720
key=54762
key=26340
key=34739
key=43099
key=41208
key=49110
key=63509
key=41898
key=29748
key=15022
key=41433
key=30777
key=42079
key=34822
key=25827
key=51245
key=61998
key=57122
key=23654
key=290
key=63313
key=48986
key=48902
key=44452
key=47510
key=51802
key=59130
key=53721
key=48923
key=58829
key=44289
key=36597
key=51517
key=52226
key=54905
key=63901
key=57445
key=38475
key=30176
key=36959
key=41865
key=48670
key=41687
key=25882
key=57722
key=47477
key=44879
key=8868
key=42879
key=33393
key=45187
key=58351
key=63908
key=49322
key=55088
key=44619
key=47935
key=37957
key=58187
key=53691
key=61736
key=215
key=44222
key=30534
key=45320
key=54691
key=449
key=44140
key=33267
key=51476
key=42496
key=56986
key=34769
key=48751
key=60434
key=56433
key=58265
key=42111
key=22094
key=60633
key=61560
key=5133
key=22605

In [197]:
result = []
for key, values in vocab_mapping_mlp.items():
    result.append(f"{key!r} — {values!r}")

with open("vocab_mapping_mlp.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

In [178]:
result

{('Ð½Ñİ', 17957): [('Ð½Ñİ', 127494)],
 ('ĠMiller', 16435): [('ĠMiller', 17035)],
 ('ĠÐ¿ÑĢÐ¸Ð½ÑĨÐ¸', 57359): [('ĠÐ¿ÑĢÐ¸Ð½', 126017), ('ÑĨÐ¸', 15107)],
 ('<|reserved_338|>', 348): [('<', 27),
  ('|r', 62640),
  ('ese', 2367),
  ('rv', 10553),
  ('ed', 291),
  ('_', 62),
  ('3', 18),
  ('3', 18),
  ('8', 23),
  ('|', 91),
  ('>', 29)],
 ('Ġtoo', 2950): [('Ġtoo', 2238)],
 ('ĠRamÃ³n', 56911): [('ĠRam', 15152), ('Ã³n', 3165)],
 ('Ġcommodities', 47071): [('Ġcommodities', 50867)],
 ('ĠÃ©s', 4156): [('ĠÃ©s', 21533)],
 ('çĽĬ', 17369): [('çĽĬ', 99419)],
 ('ellig', 6810): [('ellig', 6703)],
 ('Ġsecurity', 6368): [('Ġsecurity', 4763)],
 ('èĢ³', 39135): [('èĢ³', 100229)],
 ('ĠErinner', 59306): [('ĠErin', 55814), ('ner', 1194)],
 ('Ø§ÙĦÙħ', 20058): [('Ø§ÙĦÙħ', 124839)],
 ('ĠOptions', 38047): [('ĠOptions', 14566)],
 ('Ġinvece', 38645): [('Ġinve', 92184), ('ce', 346)],
 ('Ġstaying', 25348): [('Ġstaying', 19429)],
 ('ãĥ³ãĤ¿ãĥ«', 60503): [('ãĥ³', 15698), ('ãĤ¿ãĥ«', 133824)],
 ('ĠMay', 3354): [('ĠMay', 32

In [173]:
counter

16894

In [167]:
len(vocab_mapping_tensors)

64402

In [158]:
result

{('Ð½Ñİ', 17957): [('Ð½Ñİ', 127494)],
 ('ĠMiller', 16435): [('ĠMiller', 17035)],
 ('ĠÐ¿ÑĢÐ¸Ð½ÑĨÐ¸', 57359): [('ĠÐ¿ÑĢÐ¸Ð½', 126017), ('ÑĨÐ¸', 15107)],
 ('<|reserved_338|>', 348): [('<', 27),
  ('|r', 62640),
  ('ese', 2367),
  ('rv', 10553),
  ('ed', 291),
  ('_', 62),
  ('3', 18),
  ('3', 18),
  ('8', 23),
  ('|', 91),
  ('>', 29)],
 ('Ġtoo', 2950): [('Ġtoo', 2238)],
 ('ĠRamÃ³n', 56911): [('ĠRam', 15152), ('Ã³n', 3165)],
 ('Ġcommodities', 47071): [('Ġcommodities', 50867)],
 ('ĠÃ©s', 4156): [('ĠÃ©s', 21533)],
 ('çĽĬ', 17369): [('çĽĬ', 99419)],
 ('ellig', 6810): [('ellig', 6703)],
 ('Ġsecurity', 6368): [('Ġsecurity', 4763)],
 ('èĢ³', 39135): [('èĢ³', 100229)],
 ('ĠErinner', 59306): [('ĠErin', 55814), ('ner', 1194)],
 ('Ø§ÙĦÙħ', 20058): [('Ø§ÙĦÙħ', 124839)],
 ('ĠOptions', 38047): [('ĠOptions', 14566)],
 ('Ġinvece', 38645): [('Ġinve', 92184), ('ce', 346)],
 ('Ġstaying', 25348): [('Ġstaying', 19429)],
 ('ãĥ³ãĤ¿ãĥ«', 60503): [('ãĥ³', 15698), ('ãĤ¿ãĥ«', 133824)],
 ('ĠMay', 3354): [('ĠMay', 32

In [154]:
projector = Projector(mapping_vocab=vocab_mapping_tensors,
                      model_1=model_1,
                      model_2=model_2,
                      tokenizer_1=tokenizer_model_1,
                      tokenizer_2=tokenizer_model_2,
                      mapping_projection=result)

projector.map_input("Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?")

Разбивка по токенам: tokens_1=tensor([    1,     6,  6423,   708,   564,  1761,  8098,   592,  1058,  1374,
          902,  8831,   875,  6353,  1264,  5616,   523,  1267, 24472,   521,
         1689,  1804,  2172,   730,  1838,  5112,   803,  6353,  1264,  5616,
          523,  2213,  1972,  2172,  1689,  8098,   540,     7,   708])
Декодировано в текст: strings='user\nWeng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?\n'
tensor(1)
Токен "1" и его НС-отображение 146357
Токен "1" и его проекционное-отображение [('<', 27), ('|', 91), ('start', 2468), ('oft', 14118), ('ext', 427), ('|', 91), ('>', 29)]
tensor(6)
Токен "6" и его НС-отображение 151644
Токен "6" и его проекционное-отображение [('<|im_start|>', 151644)]
tensor(6423)
Токен "6423" и его НС-отображение 872
Токен "6423" и его проекционное-отображение [('user', 872)]
tensor(708)
Токен "708" и его НС-отображение 198
tensor(564)
Токен "564" и его НС-отображение 54
Токен

TypeError: argument 'ids': 'list' object cannot be interpreted as an integer

(('Ð½Ñİ', 17957), [('Ð½Ñİ', 127494)])


In [156]:
result

{('Ð½Ñİ', 17957): [('Ð½Ñİ', 127494)],
 ('ĠMiller', 16435): [('ĠMiller', 17035)],
 ('ĠÐ¿ÑĢÐ¸Ð½ÑĨÐ¸', 57359): [('ĠÐ¿ÑĢÐ¸Ð½', 126017), ('ÑĨÐ¸', 15107)],
 ('<|reserved_338|>', 348): [('<', 27),
  ('|r', 62640),
  ('ese', 2367),
  ('rv', 10553),
  ('ed', 291),
  ('_', 62),
  ('3', 18),
  ('3', 18),
  ('8', 23),
  ('|', 91),
  ('>', 29)],
 ('Ġtoo', 2950): [('Ġtoo', 2238)],
 ('ĠRamÃ³n', 56911): [('ĠRam', 15152), ('Ã³n', 3165)],
 ('Ġcommodities', 47071): [('Ġcommodities', 50867)],
 ('ĠÃ©s', 4156): [('ĠÃ©s', 21533)],
 ('çĽĬ', 17369): [('çĽĬ', 99419)],
 ('ellig', 6810): [('ellig', 6703)],
 ('Ġsecurity', 6368): [('Ġsecurity', 4763)],
 ('èĢ³', 39135): [('èĢ³', 100229)],
 ('ĠErinner', 59306): [('ĠErin', 55814), ('ner', 1194)],
 ('Ø§ÙĦÙħ', 20058): [('Ø§ÙĦÙħ', 124839)],
 ('ĠOptions', 38047): [('ĠOptions', 14566)],
 ('Ġinvece', 38645): [('Ġinve', 92184), ('ce', 346)],
 ('Ġstaying', 25348): [('Ġstaying', 19429)],
 ('ãĥ³ãĤ¿ãĥ«', 60503): [('ãĥ³', 15698), ('ãĤ¿ãĥ«', 133824)],
 ('ĠMay', 3354): [('ĠMay', 32